# v10 ローカル学習(LightGBM / CatBoost / XGBoost ×3)→ モデル保存

v10 の train.parquet を読み、v9 と同じパラメータで 3モデル×3シードを学習して保存する(**ローカルで実行**)。
保存した Trainer(.pkl)を Kaggle にアップロードし、提出ノート(submit_v10.ipynb)が読み込んで推論する。

- 特徴量 = parquet の全列(well/id/target を除く)= 順方向(v9)+ 逆方向(bpf_)+ 比較(cmp_)。
- CV = GroupKFold(5) by well。koolbox.Trainer で学習・保存(v9と同じ)。
- test 推論・提出はここでは行わない(Kaggle 側)。

In [1]:
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from koolbox import Trainer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from pathlib import Path
import numpy as np, pandas as pd, joblib, warnings, os, pathlib
warnings.filterwarnings("ignore")
if os.name != "nt": pathlib.WindowsPath = pathlib.PosixPath

c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CFG:
    # v10 の成果物(train.parquet があり, models/ もここに保存)
    artifacts_path = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v10\artifacts_v10_3")
    parquet = artifacts_path / "data" / "train.parquet"
    seed = 0; n_splits = 5
    cv = GroupKFold(n_splits=n_splits)
    metric = root_mean_squared_error

In [3]:
train_df = pd.read_parquet(CFG.parquet)
# 比較特徴(cmp_)は除外。順方向 + 逆方向(bpf_)のみを使用。
features = [c for c in train_df.columns if c not in {"well","id","target"} and not c.startswith("cmp_")]
X = train_df[features]; y = train_df["target"]; g = train_df["well"]
print("rows:", len(train_df), " features:", len(features), " wells:", g.nunique())

rows: 3783989  features: 511  wells: 773


In [4]:
# v9 と同じパラメータ
lgb_params = [
    dict(boosting_type="gbdt", num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, objective="regression", verbose=-1,
         n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=0, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=29, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
]
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
    dict(iterations=8000, depth=7, l2_leaf_reg=3.0, min_data_in_leaf=15, border_count=192, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.010, random_seed=999),
]
xgb_params = [
    dict(n_estimators=10000, learning_rate=0.03, max_depth=6, min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
         reg_alpha=0.1, reg_lambda=5.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=CFG.seed+0, n_jobs=-1),
    dict(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=8, subsample=0.80, colsample_bytree=0.80,
         reg_alpha=0.05, reg_lambda=4.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=CFG.seed+1, n_jobs=-1),
    dict(n_estimators=12000, learning_rate=0.02, max_depth=5, min_child_weight=25, subsample=0.75, colsample_bytree=0.75,
         reg_alpha=0.5, reg_lambda=12.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=CFG.seed+2, n_jobs=-1),
]
pp_params = {"alpha":1.0, "tau":85, "w_pf":0.09}   # 補正パラメータ(提出ノートと共有)
oof_preds = []

In [5]:
# LightGBM ×3
for i, params in enumerate(lgb_params):
    trainer = Trainer(estimator=LGBMRegressor(**params), task="regression", metric=CFG.metric, cv=CFG.cv,
                      cv_args={"groups": g}, use_early_stopping=True, verbose=True, save=True,
                      save_path=CFG.artifacts_path / f"models/lightgbm-{i+1}")
    trainer.fit(X, y, fit_args={"eval_metric":"rmse",
                "callbacks":[log_evaluation(period=250), early_stopping(stopping_rounds=250)]})
    oof_preds.append(trainer.oof_preds); print("\n")

Training LGBMRegressor

Training until validation scores don't improve for 250 rounds
[250]	valid_0's rmse: 7.3236	valid_0's l2: 53.6351
[500]	valid_0's rmse: 7.31847	valid_0's l2: 53.56
Early stopping, best iteration is:
[374]	valid_0's rmse: 7.31454	valid_0's l2: 53.5026
--- Fold 0 - root_mean_squared_error: 7.3145 - Time: 144.27 s

Training until validation scores don't improve for 250 rounds
[250]	valid_0's rmse: 7.87893	valid_0's l2: 62.0775
Early stopping, best iteration is:
[163]	valid_0's rmse: 7.87112	valid_0's l2: 61.9546
--- Fold 1 - root_mean_squared_error: 7.8711 - Time: 96.21 s

Training until validation scores don't improve for 250 rounds
[250]	valid_0's rmse: 7.24511	valid_0's l2: 52.4917
[500]	valid_0's rmse: 7.24587	valid_0's l2: 52.5026
[750]	valid_0's rmse: 7.24197	valid_0's l2: 52.4462
Early stopping, best iteration is:
[739]	valid_0's rmse: 7.24134	valid_0's l2: 52.437
--- Fold 2 - root_mean_squared_error: 7.2413 - Time: 201.60 s

Training until validation scores 

In [7]:
# CatBoost ×3
for i, params in enumerate(cb_params):
    trainer = Trainer(estimator=CatBoostRegressor(**params), task="regression", metric=CFG.metric, cv=CFG.cv,
                      cv_args={"groups": g}, use_early_stopping=True, verbose=True, save=True,
                      save_path=CFG.artifacts_path / f"models/catboost-{i+1}")
    trainer.fit(X, y, fit_args={"verbose":250, "early_stopping_rounds":250, "use_best_model":True})
    oof_preds.append(trainer.oof_preds); print("\n")

Training CatBoostRegressor

0:	learn: 15.7857618	test: 14.8603470	best: 14.8603470 (0)	total: 154ms	remaining: 20m 28s
250:	learn: 5.1853751	test: 7.7380272	best: 7.7325445 (215)	total: 11.8s	remaining: 6m 3s
500:	learn: 4.0794664	test: 7.7192925	best: 7.7073045 (383)	total: 47.6s	remaining: 11m 52s
bestTest = 7.707304511
bestIteration = 383
Shrink model to first 384 iterations.
--- Fold 0 - root_mean_squared_error: 7.7073 - Time: 85.14 s

0:	learn: 15.6272933	test: 15.5456530	best: 15.5456530 (0)	total: 82.7ms	remaining: 11m 1s
250:	learn: 5.1646082	test: 7.6064335	best: 7.6056336 (249)	total: 16.1s	remaining: 8m 16s
500:	learn: 4.0788704	test: 7.5876847	best: 7.5639576 (444)	total: 32.1s	remaining: 8m 1s
bestTest = 7.563957565
bestIteration = 444
Shrink model to first 445 iterations.
--- Fold 1 - root_mean_squared_error: 7.5640 - Time: 88.58 s

0:	learn: 15.8507199	test: 14.5839560	best: 14.5839560 (0)	total: 51.5ms	remaining: 6m 51s
250:	learn: 5.1519394	test: 7.3255832	best: 7.3255

In [8]:
# XGBoost ×3
for i, params in enumerate(xgb_params):
    pr = dict(params); pr.setdefault("early_stopping_rounds", 250)
    trainer = Trainer(estimator=XGBRegressor(**pr), task="regression", metric=CFG.metric, cv=CFG.cv,
                      cv_args={"groups": g}, use_early_stopping=True, verbose=True, save=True,
                      save_path=CFG.artifacts_path / f"models/xgboost-{i+1}")
    trainer.fit(X, y, fit_args={"verbose":250})
    oof_preds.append(trainer.oof_preds); print("\n")

Training XGBRegressor

[0]	validation_0-rmse:14.75502
[250]	validation_0-rmse:7.44488
[500]	validation_0-rmse:7.44733
[594]	validation_0-rmse:7.44437
--- Fold 0 - root_mean_squared_error: 7.4296 - Time: 87.20 s

[0]	validation_0-rmse:15.43743
[250]	validation_0-rmse:7.70886
[367]	validation_0-rmse:7.72434
--- Fold 1 - root_mean_squared_error: 7.6847 - Time: 70.00 s

[0]	validation_0-rmse:14.47028
[250]	validation_0-rmse:7.33670
[500]	validation_0-rmse:7.28540
[750]	validation_0-rmse:7.27288
[1000]	validation_0-rmse:7.26963
[1250]	validation_0-rmse:7.26293
[1455]	validation_0-rmse:7.26551
--- Fold 2 - root_mean_squared_error: 7.2609 - Time: 106.61 s

[0]	validation_0-rmse:15.74070
[250]	validation_0-rmse:7.90228
[350]	validation_0-rmse:7.93841
--- Fold 3 - root_mean_squared_error: 7.7798 - Time: 81.61 s

[0]	validation_0-rmse:17.00175
[250]	validation_0-rmse:9.60577
[500]	validation_0-rmse:9.50976
[750]	validation_0-rmse:9.50112
[1000]	validation_0-rmse:9.44116
[1250]	validation_0-rmse:

In [ ]:
print(f"Final OOF RMSE (9-model mean): {CFG.metric(np.mean(oof_preds, axis=0), y):.4f}")
print("models saved under:", CFG.artifacts_path / "models")
print("→ この models/ フォルダ(と train.parquet は不要)を Kaggle データセットにアップロードして submit_v10.ipynb で使用")